# LangGraph

## Library

In [21]:
import os
import sys
from pathlib import Path

import requests
from dotenv import load_dotenv
from langchain.agents import create_agent
from langsmith import Client

import urllib.error
import urllib.request

from langchain.tools import tool
from deepagents import create_deep_agent
from langchain.chat_models import init_chat_model
from IPython.display import Markdown, display
from langchain.messages import HumanMessage, AIMessage, SystemMessage
from langchain.tools import tool

from pydantic import BaseModel, Field
import ch_charges as ch_ch
import ch_filing_history as ch_fh
import json
from IPython.display import Image, display

from langchain.messages import AnyMessage
from typing_extensions import TypedDict, Annotated, Literal
import operator
from langgraph.graph import StateGraph, MessagesState, START, END
load_dotenv() # looks for a .env file in the current or parent directions

api_key = os.getenv('ANTHROPIC_API_KEY')
langsmith_api_key = os.getenv('LANGSMITH_API_KEY')

## Hello World

In [13]:


def mock_llm(state: MessagesState):
    return {"messages": [{"role": "ai", "content": "hello world"}]}

graph = StateGraph(MessagesState)
graph.add_node(mock_llm)
graph.add_edge(START, "mock_llm")
graph.add_edge("mock_llm", END)
graph = graph.compile()

graph.invoke({"messages": [{"role": "user", "content": "hi!"}]})



{'messages': [HumanMessage(content='hi!', additional_kwargs={}, response_metadata={}, id='237572f0-d4e2-4aa9-bd5b-4346c59d06de'),
  AIMessage(content='hello world', additional_kwargs={}, response_metadata={}, id='3943b912-97e4-4d95-8400-3e2ece683d61', tool_calls=[], invalid_tool_calls=[])]}

## Calculator with LangGraph

### Define tools and model

In [14]:


model = init_chat_model(
    'claude-sonnet-4-6',
    temperature = 0
)

# Define tools
@tool
def multiply(a: int, b: int) -> int:
    """Multiply 'a' and 'b'.
    
    Args:
        a: First int
        b: Second int
    
    """
    return a * b

@tool
def add(a: int, b: int) -> int:
    """Adds 'a' and 'b'.
    
    Args:
        a: First int
        b: Second int
    """
    return a + b

@tool
def divide(a: int, b: int) -> float:
    """Divide 'a' and 'b'.
    
    Args:
        a: First int
        b: Second int
    """
    return a / b

# Augment the LLM with tools
tools = [add, multiply, divide]
tools_by_name = {tool.name: tool for tool in tools}
model_with_tools = model.bind_tools(tools)

### Define state

The graph's state is used to store the messages and the number of LLM calls

In [ ]:


class MessagesState(TypedDict):
    messages: Annotated[list[AnyMessage], operator.add]
    llm_calls: int

### Define model node

The model node is used to call the LLM and decide whether to call a tool or not

In [16]:
from langchain.messages import SystemMessage

def llm_call(state: dict):
    """LLM decides whether to call a tool or not"""
    
    return {
        "messages": [
            model_with_tools.invoke(
                [
                    SystemMessage(
                        content = "You are a helpful assistant tasked with performing arithmetic on a set of inputs."
                    )
                ]
                + state["messages"]
            )
        ],
        "llm_calls": state.get('llm_calls', 0) + 1
    }

### Define tool node

The tool node is used to call the tools and return the results

In [17]:
from langchain.messages import ToolMessage

def tool_node(state: dict):
    """Performs the tool call"""
    
    result = []
    for tool_call in state["messages"][-1].tool_calls:
        tool = tools_by_name[tool_call["name"]]
        observation = tool.invoke(tool_call["args"])
        result.append(ToolMessage(content = observation, tool_call_id = tool_call["id"]))
    return {"messages": result}

### Define end logic

The conditional edge function is used to route to the tool node or end based upon whether the LLM made a tool call.

In [18]:


def should_continue(state: MessagesState) -> Literal["tool_node", END]:
    """Decide if we should continue the loop or stop based upon whether the LLM made a tool call"""
    
    messages = state["messages"]
    last_message = messages[-1]
    
    # If the LLM makes a tool call, then perform an action
    if last_message.tool_calls:
        return "tool_node"
    
    # Otherwise, we stop (reply to the user)
    return END

### Build and compile the agent

The agent is built using the StateGraph class and compiled using the compile method

In [20]:
# Build workflow
agent_builder = StateGraph(MessagesState)

# Add nodes
agent_builder.add_node("llm_call", llm_call)
agent_builder.add_node("tool_node", tool_node)

# Add edges to connect nodes
agent_builder.add_edge(START, "llm_call")
agent_builder.add_conditional_edges(
    "llm_call",
    should_continue,
    ["tool_node", END]
)
agent_builder.add_edge("tool_node", "llm_call")

# Compile the agent
agent = agent_builder.compile()

# Show the agent
display(Image(agent.get_graph(xray = True).draw_mermaid()))

# Invoke
messages = [HumanMessage(content = "Add 3 and 4.")]
messages = agent.invoke({"messages": messages})

for m in messages["messages"]:
    m.pretty_print()

FileNotFoundError: No such file or directory: '---
config:
  flowchart:
    curve: linear
---
graph TD;
	__start__([<p>__start__</p>]):::first
	llm_call(llm_call)
	tool_node(tool_node)
	__end__([<p>__end__</p>]):::last
	__start__ --> llm_call;
	llm_call -.-> __end__;
	llm_call -.-> tool_node;
	tool_node --> llm_call;
	classDef default fill:#f2f0ff,line-height:1.2
	classDef first fill-opacity:0
	classDef last fill:#bfb6fc
'

FileNotFoundError: No such file or directory: '---
config:
  flowchart:
    curve: linear
---
graph TD;
	__start__([<p>__start__</p>]):::first
	llm_call(llm_call)
	tool_node(tool_node)
	__end__([<p>__end__</p>]):::last
	__start__ --> llm_call;
	llm_call -.-> __end__;
	llm_call -.-> tool_node;
	tool_node --> llm_call;
	classDef default fill:#f2f0ff,line-height:1.2
	classDef first fill-opacity:0
	classDef last fill:#bfb6fc
'

<IPython.core.display.Image object>

================================ Human Message =================================

Add 3 and 4.
================================== Ai Message ==================================

[{'id': 'toolu_01MSDkHWmKPbU4YkLMf1jxZk', 'caller': {'type': 'direct'}, 'input': {'a': 3, 'b': 4}, 'name': 'add', 'type': 'tool_use', 'toolset_name': None}]
Tool Calls:
  add (toolu_01MSDkHWmKPbU4YkLMf1jxZk)
 Call ID: toolu_01MSDkHWmKPbU4YkLMf1jxZk
  Args:
    a: 3
    b: 4
================================= Tool Message =================================

7
================================== Ai Message ==================================

The sum of 3 and 4 is **7**.
